# Solution: MySQL Installation, Workbench Connection, and Product CRUD

This notebook provides the complete solution for the class activity.

It assumes that:

1. MySQL Server is already installed.
2. MySQL Workbench is installed.
3. The student can connect to MySQL Workbench using the `root` account.
4. The student knows the password created during MySQL installation.


## Step 1: Install MySQL Connector for Python

In [ ]:
!pip install mysql-connector-python

## Step 2: Import Required Library

In [ ]:
import mysql.connector
from mysql.connector import Error

## Step 3: Define MySQL Credentials

Replace the password below with the password created for the local MySQL `root` account.


In [ ]:
MYSQL_HOST = "localhost"
MYSQL_USER = "root"
MYSQL_PASSWORD = "YOUR_PASSWORD_HERE"  # Replace with your own root password
DATABASE_NAME = "college_activity" 

## Step 4: Connect to the MySQL Server

In [ ]:
def connect_to_mysql_server():
    """Connect to the local MySQL Server without selecting a database yet."""
    try:
        connection = mysql.connector.connect(
            host=MYSQL_HOST,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD
        )

        if connection.is_connected():
            print("Successfully connected to MySQL Server.")
            return connection

    except Error as e:
        print("Connection failed.")
        print("Error:", e)
        return None


server_connection = connect_to_mysql_server()

## Step 5: Create the Database

In [ ]:
def create_database(connection, database_name):
    """Create a database if it does not already exist."""
    cursor = connection.cursor()
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {database_name}")
    cursor.close()
    print(f"Database '{database_name}' is ready.")


create_database(server_connection, DATABASE_NAME)

## Step 6: Connect to the Database

In [ ]:
def connect_to_database():
    """Connect directly to the selected MySQL database."""
    try:
        connection = mysql.connector.connect(
            host=MYSQL_HOST,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD,
            database=DATABASE_NAME
        )

        if connection.is_connected():
            print(f"Successfully connected to database '{DATABASE_NAME}'.")
            return connection

    except Error as e:
        print("Database connection failed.")
        print("Error:", e)
        return None


if server_connection and server_connection.is_connected():
    server_connection.close()
    print("Server-level connection closed.")

db_connection = connect_to_database()

## Step 7: Create the `products` Table

In [ ]:
def create_products_table(connection):
    """Create the products table if it does not already exist."""
    cursor = connection.cursor()

    create_table_query = """
    CREATE TABLE IF NOT EXISTS products (
        product_id INT AUTO_INCREMENT PRIMARY KEY,
        product_name VARCHAR(100) NOT NULL,
        category VARCHAR(100),
        price DECIMAL(10, 2) NOT NULL,
        quantity INT NOT NULL
    )
    """

    cursor.execute(create_table_query)
    cursor.close()
    print("Table 'products' is ready.")


create_products_table(db_connection)

## Step 8: Optional Reset for Repeatable Classroom Demonstration

This makes the notebook easier to rerun during class by clearing old records and resetting the auto-increment counter.


In [ ]:
def reset_products_table(connection):
    """Remove all records and reset the product_id counter."""
    cursor = connection.cursor()
    cursor.execute("TRUNCATE TABLE products")
    connection.commit()
    cursor.close()
    print("Products table has been reset.")


reset_products_table(db_connection)

## Step 9: CREATE Operation: Insert Products

In [ ]:
def insert_product(connection, product_name, category, price, quantity):
    """Insert one product into the products table."""
    cursor = connection.cursor()

    insert_query = """
    INSERT INTO products (product_name, category, price, quantity)
    VALUES (%s, %s, %s, %s)
    """

    product_data = (product_name, category, price, quantity)

    cursor.execute(insert_query, product_data)
    connection.commit()

    print(f"Inserted product: {product_name}")

    cursor.close()


insert_product(db_connection, "Laptop", "Electronics", 999.99, 10)
insert_product(db_connection, "Notebook", "Stationery", 4.99, 100)
insert_product(db_connection, "Coffee Mug", "Kitchen", 12.50, 25)

## Step 10: READ Operation: Display Products

In [ ]:
def read_products(connection):
    """Read and display all products."""
    cursor = connection.cursor()

    select_query = """
    SELECT product_id, product_name, category, price, quantity
    FROM products
    ORDER BY product_id
    """

    cursor.execute(select_query)
    rows = cursor.fetchall()

    if len(rows) == 0:
        print("No products found.")
    else:
        print("Current products:")
        for row in rows:
            print(row)

    cursor.close()


read_products(db_connection)

## Step 11: UPDATE Operation: Update Product Price and Quantity

In this example, we update the product with `product_id = 1`.


In [ ]:
def update_product(connection, product_id, new_price, new_quantity):
    """Update the price and quantity for one product."""
    cursor = connection.cursor()

    update_query = """
    UPDATE products
    SET price = %s, quantity = %s
    WHERE product_id = %s
    """

    cursor.execute(update_query, (new_price, new_quantity, product_id))
    connection.commit()

    print(f"Updated product with ID: {product_id}")

    cursor.close()


update_product(db_connection, 1, 899.99, 8)
read_products(db_connection)

## Step 12: DELETE Operation: Delete One Product

In this example, we delete the product with `product_id = 2`.


In [ ]:
def delete_product(connection, product_id):
    """Delete one product using its product_id."""
    cursor = connection.cursor()

    delete_query = """
    DELETE FROM products
    WHERE product_id = %s
    """

    cursor.execute(delete_query, (product_id,))
    connection.commit()

    print(f"Deleted product with ID: {product_id}")

    cursor.close()


delete_product(db_connection, 2)
read_products(db_connection)

## Step 13: Full CRUD Demonstration

This section completes a full CRUD cycle on one new product.


In [ ]:
# CREATE
insert_product(db_connection, "Wireless Mouse", "Electronics", 24.99, 40)

# READ
read_products(db_connection)

# UPDATE
# Since this notebook resets the table, Wireless Mouse should receive product_id = 4.
update_product(db_connection, 4, 19.99, 35)

# READ again
read_products(db_connection)

# DELETE
delete_product(db_connection, 4)

# Final READ
read_products(db_connection)

## Step 14: Delete All Products

This function removes every product from the table. Use carefully.


In [ ]:
def delete_all_products(connection):
    """Delete all records from the products table."""
    cursor = connection.cursor()
    cursor.execute("DELETE FROM products")
    connection.commit()
    cursor.close()
    print("All products were deleted.")


# Uncomment if needed:
# delete_all_products(db_connection)
# read_products(db_connection)

## Step 15: Close the Database Connection

In [ ]:
if db_connection and db_connection.is_connected():
    db_connection.close()
    print("Database connection closed.")

## Instructor Notes

Expected successful output should show:

1. A successful MySQL Server connection.
2. Creation or confirmation of the `college_activity` database.
3. Creation or confirmation of the `products` table.
4. Three inserted products.
5. Updated price and quantity for one product.
6. Deletion of one product.
7. A complete CREATE, READ, UPDATE, DELETE cycle for `Wireless Mouse`.

Common issues:

- `Access denied for user 'root'@'localhost'`: incorrect password.
- `Can't connect to MySQL server`: MySQL Server is not running.
- `ModuleNotFoundError: No module named 'mysql'`: connector was not installed in the active Python/Jupyter environment.
- `Unknown database`: database creation cell was not executed.
